<a href="https://colab.research.google.com/github/guillaumevalette2-hash/mse_gh/blob/main/multi_pol_bezrbf.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import numpy as np
from itertools import product
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_squared_error
import time

params = {
    "n_ambiant": 4, "deg_P": 3, "n_terms_poly": 20,
   "seeds": {1: 312711, 2: 437346761, 3: 7455242627531,
              42:45982467543, 43:7335535176, 44: 4542372435, 11:1235431335, 12: 33237838871},
    "weights":   {0: 0, 1: 1, 2: 0.1, 3: 0.01},
    "n_train":   20,
    "n_test":    5000,
    "n_levels":  5,       # nb de fonctions f_i cylindriques
    "n_levels_iso": 5,    # nb de fonctions f_i isotropes (sigma unique par centre)
    "k_loss":    3.0,     # sélection finale : garder les experts avec loss <= k_loss * meilleure loss
    "deg_poly_expert": 6,      # degré de l'expert polynomial (dérivées analytiques exactes)
    "n_dict":    5000,     # taille dictionnaire par niveau
    "n_centres": 800,      # centres retenus par niveau
    "sigma_min": 0.1,      # borne inférieure des sigma_i
    "sigma_max": 3.0,      # borne supérieure des sigma_i
    "lambda_reg": 1e-5,    # UNIQUE régularisation Sobolev (Phase 1, sélection, Phase 2) :
                           # la norme Sobolev est indépendante de la base -> une seule loss
    "n_G":       2000,      # points pour G de chaque f_i
    "n_G_H":     5000,     # points pour la Gram finale sur H
}
params["n_unlabeled"]        = 4000 - params["n_train"]
params["train_center_ratio"] = 0.5 + params["n_train"] / (2 * 4000)

# ══════════════════════════════════════════════════════════════════════════════
# POLYNÔMES ET CLOUD
# ══════════════════════════════════════════════════════════════════════════════
def random_sparse_polynomial(d, degree, n_terms, seed=None):
    rng = np.random.default_rng(seed)
    all_indices = [exp for exp in product(range(degree+1), repeat=d)
                   if sum(exp) <= degree]
    indices = rng.choice(all_indices, size=n_terms, replace=False)
    coeffs  = rng.normal(size=n_terms)
    def P(x):
        y = np.zeros(x.shape[0])
        for c, alpha in zip(coeffs, indices):
            term = np.ones(x.shape[0])
            for j, e in enumerate(alpha):
                if e > 0: term *= x[:,j]**e
            y += c * term
        return y
    zero_val = sum(c for c, alpha in zip(coeffs, indices)
                   if all(e == 0 for e in alpha))
    def P_zero(x): return P(x) - zero_val
    return P_zero, indices, coeffs

def normalize_polynomial(P, indices, coeffs):
    max_c = np.max(np.abs(coeffs))
    if max_c > 0:
        nc = coeffs / max_c
        def Pn(x):
            y = np.zeros(x.shape[0])
            for c, alpha in zip(nc, indices):
                term = np.ones(x.shape[0])
                for j, e in enumerate(alpha):
                    if e > 0: term *= x[:,j]**e
                y += c * term
            return y
        return Pn
    return P

P1,i1,c1 = random_sparse_polynomial(params["n_ambiant"],params["deg_P"],params["n_terms_poly"],params["seeds"][1])
P2,i2,c2 = random_sparse_polynomial(params["n_ambiant"],params["deg_P"],params["n_terms_poly"],params["seeds"][2])
P3,i3,c3 = random_sparse_polynomial(params["n_ambiant"],params["deg_P"],params["n_terms_poly"],params["seeds"][3])
P1=normalize_polynomial(P1,i1,c1); P2=normalize_polynomial(P2,i2,c2); P3=normalize_polynomial(P3,i3,c3)
def Q(x): return P1(x)*P2(x)*P3(x)

def grad_Q_analytical(X):
    eps=1e-5; d=X.shape[1]; p1=P1(X); p2=P2(X); p3=P3(X)
    grad=np.zeros_like(X)
    for k in range(d):
        Xp=X.copy(); Xp[:,k]+=eps; Xm=X.copy(); Xm[:,k]-=eps
        dp1=(P1(Xp)-P1(Xm))/(2*eps); dp2=(P2(Xp)-P2(Xm))/(2*eps); dp3=(P3(Xp)-P3(Xm))/(2*eps)
        grad[:,k]=dp1*p2*p3+p1*dp2*p3+p1*p2*dp3
    return grad

def project_to_Q_zero(X_init, n_steps=150, lr=0.05, tol=1e-4):
    X=X_init.copy()
    for _ in range(n_steps):
        q=Q(X); gq=grad_Q_analytical(X); gq2=2*q[:,None]*gq
        norm=np.linalg.norm(gq2,axis=1,keepdims=True)+1e-12
        X=X-lr*gq2/norm; X=np.clip(X,0.,1.)
        if np.abs(Q(X)).max()<tol*0.1: break
    return X, np.abs(Q(X))<tol

def sample_on_Q_zero(n_target, d, seed=None):
    rng=np.random.default_rng(seed); collected=[]; n_col=0
    while n_col<n_target:
        n_batch=max((n_target-n_col)*4,200)
        Xi=rng.uniform(0,1,(int(n_batch),d)); Xp,conv=project_to_Q_zero(Xi)
        good=Xp[conv]
        if len(good)>0: collected.append(good); n_col+=len(good)
        print(f"  collectés:{n_col}/{n_target} ({conv.mean():.1%})",end='\r')
    print(); return np.vstack(collected)[:n_target]

print("Génération du cloud sur {Q=0}...")
d=params["n_ambiant"]
X_train     = sample_on_Q_zero(params["n_train"],    d, seed=params["seeds"][42])
X_test      = sample_on_Q_zero(params["n_test"],     d, seed=params["seeds"][43])
X_unlabeled = sample_on_Q_zero(params["n_unlabeled"],d, seed=params["seeds"][44])
print(f"  X_train:{X_train.shape}  X_unlabeled:{X_unlabeled.shape}")

P_target1, indicest1, coeffst1 = random_sparse_polynomial(params["n_ambiant"],
                                           4, params["n_terms_poly"], seed=params["seeds"][11])
P_target2, _, _ = random_sparse_polynomial(params["n_ambiant"],
                                           4, params["n_terms_poly"], seed=params["seeds"][12])
Ptarget1 = normalize_polynomial(P_target1, indicest1, coeffst1)
Ptarget2 = normalize_polynomial(P_target2, indicest1, coeffst1)

def target_function(X):
    return np.abs(P_target1(X)) + np.abs(P_target2(X))

y_train=target_function(X_train); y_test=target_function(X_test)
X_all=np.vstack([X_train,X_unlabeled]); N_all=len(X_all)

# ══════════════════════════════════════════════════════════════════════════════
# GAUSSIENNES CYLINDRIQUES
# ══════════════════════════════════════════════════════════════════════════════
def sample_candidates(X_train, X_unlabeled, n_candidates, train_ratio, rng):
    n_tr=min(int(round(n_candidates*train_ratio)),X_train.shape[0])
    n_ul=min(n_candidates-n_tr,X_unlabeled.shape[0]); parts=[]
    if n_tr>0: parts.append(X_train[rng.choice(X_train.shape[0],n_tr,replace=False)])
    if n_ul>0: parts.append(X_unlabeled[rng.choice(X_unlabeled.shape[0],n_ul,replace=False)])
    centers=np.vstack(parts)
    log_s=rng.uniform(np.log(params["sigma_min"]),np.log(params["sigma_max"]),
                      size=(len(centers),d))
    sigmas=np.exp(log_s)
    return centers, sigmas

def sample_candidates_iso(X_train, X_unlabeled, n_candidates, train_ratio, rng):
    """Gaussiennes isotropes : un seul sigma par centre, répété sur les d dimensions.
    Cas particulier des cylindriques -> tout le pipeline (features, dérivées, Gram)
    fonctionne sans modification."""
    n_tr=min(int(round(n_candidates*train_ratio)),X_train.shape[0])
    n_ul=min(n_candidates-n_tr,X_unlabeled.shape[0]); parts=[]
    if n_tr>0: parts.append(X_train[rng.choice(X_train.shape[0],n_tr,replace=False)])
    if n_ul>0: parts.append(X_unlabeled[rng.choice(X_unlabeled.shape[0],n_ul,replace=False)])
    centers=np.vstack(parts)
    log_s=rng.uniform(np.log(params["sigma_min"]),np.log(params["sigma_max"]),
                      size=len(centers))                      # UN sigma par centre
    sigmas=np.tile(np.exp(log_s)[:,None],(1,d))               # répété sur les d dims
    return centers, sigmas

def cyl_features(X, centers, sigmas):
    diff=X[:,None,:]-centers[None,:,:]
    sq_w=np.sum(diff**2/sigmas[None,:,:]**2, axis=2)
    return np.exp(-sq_w)

def cyl_derivatives(X, centers, sigmas, weights=None):
    d_=X.shape[1]; n=X.shape[0]; m=centers.shape[0]
    diff=X[:,None,:]-centers[None,:,:]
    inv_s2=1./sigmas**2
    diff_w=diff*inv_s2[None,:,:]
    sq_w=np.sum(diff*diff_w, axis=2)
    phi=np.exp(-sq_w)
    grad=-2*diff_w*phi[:,:,None]
    need_hess  = weights is None or weights.get(2,0.)!=0 or weights.get(3,0.)!=0
    need_third = weights is None or weights.get(3,0.)!=0
    if need_hess:
        Id=np.eye(d_)
        diag=-2*inv_s2[None,:,:,None]*Id[None,None,:,:]
        cross=4*np.einsum('nmi,nmj->nmij',diff_w,diff_w)
        hess=(diag+cross)*phi[:,:,None,None]
    else: hess=None
    if need_third:
        Id=np.eye(d_)
        inv_s2_bc=inv_s2[None,:,:]
        cubic=-8*np.einsum('nmi,nmj,nmk->nmijk',diff_w,diff_w,diff_w)
        sym=4*(np.einsum('ij,nmi,nmk->nmijk',Id,inv_s2_bc,diff_w)+
               np.einsum('ik,nmi,nmj->nmijk',Id,inv_s2_bc,diff_w)+
               np.einsum('jk,nmj,nmi->nmijk',Id,inv_s2_bc,diff_w))
        third=(cubic+sym)*phi[:,:,None,None,None]
    else: third=None
    return phi, grad, hess, third

def build_G(phi, grad, hess, third, weights, n):
    G=np.zeros((phi.shape[1],phi.shape[1]))
    w0=weights.get(0,0.)
    if w0: G+=w0*(phi.T@phi)/n
    w1=weights.get(1,0.)
    if w1 and grad  is not None: G+=w1*np.einsum('xik,xjk->ij',    grad, grad, optimize='optimal')/n
    w2=weights.get(2,0.)
    if w2 and hess  is not None: G+=w2*np.einsum('xikl,xjkl->ij',  hess, hess, optimize='optimal')/n
    w3=weights.get(3,0.)
    if w3 and third is not None: G+=w3*np.einsum('xiklm,xjklm->ij',third,third,optimize='optimal')/n
    return G

# ══════════════════════════════════════════════════════════════════════════════
# PHASE 1 : calculer n_levels fonctions f_i indépendantes
# Chaque f_i = argmin sur son propre dictionnaire cylindrique aléatoire
# ══════════════════════════════════════════════════════════════════════════════
level_specs = ([("cyl", i) for i in range(params["n_levels"])] +
               [("iso", i) for i in range(params["n_levels_iso"])])
n_levels_total = len(level_specs)
expert_names   = [f"{typ}_{i+1:02d}" for typ, i in level_specs]

print(f"\nPhase 1 : calcul de {n_levels_total} fonctions indépendantes "
      f"({params['n_levels']} cyl + {params['n_levels_iso']} iso)...")

F_train = np.zeros((len(X_train), n_levels_total))   # f_i(X_train)
F_test  = np.zeros((len(X_test),  n_levels_total))   # f_i(X_test)
F_all   = np.zeros((N_all,        n_levels_total))   # f_i(X_all)  pour Gram H

# Pour la Gram finale on a besoin des dérivées de chaque f_i sur n_G_H points
# On les stocke compressées : delta_phi/grad/hess/third = fi_derivs sur X_all
fi_derivs_list = []  # liste de dicts par niveau

times_p1 = []
for level, (lev_type, lev_idx) in enumerate(level_specs):
    t0 = time.time()
    # graines distinctes cyl / iso pour éviter les mêmes centres
    rng = np.random.default_rng(seed=lev_idx if lev_type=="cyl" else 1000+lev_idx)

    sampler = sample_candidates if lev_type=="cyl" else sample_candidates_iso
    candidates, sigmas_cand = sampler(
        X_train, X_unlabeled, params["n_dict"], params["train_center_ratio"], rng)

    # Score par corrélation avec y_train
    A_cand = cyl_features(X_train, candidates, sigmas_cand)
    corr   = (A_cand.T @ y_train) / len(X_train)
    scores = corr**2

    k     = min(params["n_centres"], len(candidates))
    top_k = np.argsort(scores)[-k:]
    centers_sel = candidates[top_k]
    sigmas_sel  = sigmas_cand[top_k]

    A     = A_cand[:, top_k]
    Atest = cyl_features(X_test, centers_sel, sigmas_sel)
    Aall  = cyl_features(X_all,  centers_sel, sigmas_sel)

    phi, grad, hess, third = cyl_derivatives(
        X_all, centers_sel, sigmas_sel, params["weights"])

    n_G   = min(N_all, params["n_G"])
    idx_G = rng.choice(N_all, size=n_G, replace=False)
    G = build_G(phi[idx_G],
                grad[idx_G]  if grad  is not None else None,
                hess[idx_G]  if hess  is not None else None,
                third[idx_G] if third is not None else None,
                params["weights"], n_G)

    lambda_reg = params["lambda_reg"]
    n = A.shape[0]
    M   = (A.T@A)/n + lambda_reg*G
    rhs = (A.T@y_train)/n

    eigvals,eigvecs = np.linalg.eigh(M)
    thresh = max(lambda_reg, 0); mask = eigvals > thresh
    V=eigvecs[:,mask]; S=eigvals[mask]; coeffs=V@((V.T@rhs)/S)

    F_train[:,level] = A    @ coeffs
    F_test [:,level] = Atest @ coeffs
    F_all  [:,level] = Aall  @ coeffs

    # Dérivées de f_i sur X_all pour la Gram finale
    fi_derivs_list.append({
        'phi':   phi   @ coeffs,
        'grad':  np.einsum('xjk,j->xk',   grad, coeffs) if grad  is not None else None,
        'hess':  np.einsum('xjkl,j->xkl', hess, coeffs) if hess  is not None else None,
        'third': np.einsum('xjklm,j->xklm',third,coeffs) if third is not None else None,
    })

    loss_data  = np.mean((y_train - A @ coeffs)**2)
    loss_reg   = lambda_reg * coeffs @ G @ coeffs
    loss_total = loss_data + loss_reg
    mse_i = mean_squared_error(y_test, F_test[:,level])
    t1=time.time(); times_p1.append(t1-t0)
    print(f"  {expert_names[level]:7s} | rang={mask.sum():3d}/{k} | "
          f"loss={loss_total:.6f} (data={loss_data:.6f} reg={loss_reg:.6f}) | "
          f"MSE={mse_i:.6f} | σ∈[{sigmas_sel.min():.2f},{sigmas_sel.max():.2f}] | "
          f"{times_p1[-1]:.1f}s")


# ══════════════════════════════════════════════════════════════════════════════
# AJOUT DES EXPERTS RBF ET RIDGE AVEC DÉRIVÉES SOBOLEV SANS TENSEUR COMPLET
# ══════════════════════════════════════════════════════════════════════════════
print("\nCalibration expert polynomial...")
from sklearn.preprocessing import PolynomialFeatures

# ── Sous-échantillon pour les dérivées (cohérent avec la Gram) ────────────────
rng_H = np.random.default_rng(seed=999)
idx_H = rng_H.choice(N_all, size=min(N_all, params["n_G_H"]), replace=False)
n_H   = len(idx_H)
X_H   = X_all[idx_H]   # (n_H, d)

w1 = params["weights"].get(1, 0.)
w2 = params["weights"].get(2, 0.)
w3 = params["weights"].get(3, 0.)

# ── Expert polynomial degré deg_poly_expert (dérivées analytiques exactes) ───
deg_e   = params["deg_poly_expert"]
d_      = params["n_ambiant"]
poly_e  = PolynomialFeatures(degree=deg_e, include_bias=False)
Phi_tr  = poly_e.fit_transform(X_train)
ridge_e = Ridge(alpha=1e-8)             # petite régularisation numérique du fit
ridge_e.fit(Phi_tr, y_train)
powers_e    = poly_e.powers_            # (n_feat, d) : exposants de chaque monôme
coefs_e     = ridge_e.coef_.copy()
intercept_e = float(ridge_e.intercept_)

pred_poly_train = ridge_e.predict(Phi_tr)
pred_poly_test  = ridge_e.predict(poly_e.transform(X_test))
print(f"  Poly{deg_e} expert : MSE={mean_squared_error(y_test, pred_poly_test):.6f}")

def poly_deriv_eval(X, powers, coefs, dims=()):
    """Évaluation analytique de la dérivée d'ordre len(dims) du polynôme
    f(x) = Σ_j c_j Π_k x_k^{p_jk} par rapport aux dimensions listées dans
    dims (avec multiplicité). Exacte : aucune différence finie."""
    P = powers.astype(np.float64).copy()
    mult = coefs.astype(np.float64).copy()
    for m in dims:
        mult = mult * P[:, m]     # facteur p (p-1) ... et annulation exacte si p=0
        P[:, m] -= 1
    valid = mult != 0
    if not np.any(valid):
        return np.zeros(len(X))
    Xp = X[:, None, :] ** P[None, valid, :]        # (n, n_valid, d)
    return Xp.prod(axis=2) @ mult[valid]

delta_phi_poly = poly_deriv_eval(X_H, powers_e, coefs_e) + intercept_e

if w1 or w2 or w3:
    delta_grad_poly = np.stack(
        [poly_deriv_eval(X_H, powers_e, coefs_e, (k_,)) for k_ in range(d_)], axis=1)
else:
    delta_grad_poly = None

if w2 or w3:
    delta_hess_poly = np.zeros((n_H, d_, d_))
    for k_ in range(d_):
        for l_ in range(k_, d_):
            v = poly_deriv_eval(X_H, powers_e, coefs_e, (k_, l_))
            delta_hess_poly[:,k_,l_] = v; delta_hess_poly[:,l_,k_] = v
else:
    delta_hess_poly = None

if w3:
    delta_third_poly = np.zeros((n_H, d_, d_, d_))
    from itertools import permutations
    for k_ in range(d_):
        for l_ in range(k_, d_):
            for m_ in range(l_, d_):
                v = poly_deriv_eval(X_H, powers_e, coefs_e, (k_, l_, m_))
                for a,b,c in set(permutations((k_,l_,m_))):
                    delta_third_poly[:,a,b,c] = v
else:
    delta_third_poly = None

# ── Ajout aux F et fi_derivs ──────────────────────────────────────────────────
F_train = np.column_stack([F_train, pred_poly_train])
F_test  = np.column_stack([F_test,  pred_poly_test])

fi_derivs_list.append({
    'phi': delta_phi_poly,   'grad': delta_grad_poly,
    'hess': delta_hess_poly, 'third': delta_third_poly})

expert_names += [f"Poly{deg_e}"]
print(f"  H étendu à {len(fi_derivs_list)} fonctions")

# ══════════════════════════════════════════════════════════════════════════════
# PHASE 2 : argmin sur H = <f_1,...,f_n_levels>
# On cherche α* = argmin_{α} ||y - F_train α||² + λ_H α^T G_H α
# où G_H_{ij} = <f_i, f_j>_Sobolev calculé sur n_G_H points
# ══════════════════════════════════════════════════════════════════════════════
print(f"\nPhase 2 : Gram finale sur H ({len(fi_derivs_list)} fonctions, n_G_H={n_H})...")

# idx_H et n_H déjà définis dans le bloc experts
# Gram de H : G_H[i,j] = <f_i, f_j>_Sobolev sur idx_H
n_lev = len(fi_derivs_list)
G_H = np.zeros((n_lev, n_lev))
w0=params["weights"].get(0,0.); w1=params["weights"].get(1,0.)
w2=params["weights"].get(2,0.); w3=params["weights"].get(3,0.)

# phi de chaque f_i sur idx_H : shape (n_H,)
if w0:
    PHI_H = np.stack([
        fi['phi'] if fi['phi'].shape[0] == n_H else fi['phi'][idx_H]
        for fi in fi_derivs_list], axis=1)
    G_H += w0*(PHI_H.T@PHI_H)/n_H

if w1:
    GRAD_H = np.stack([
        fi['grad'] if fi['grad'].shape[0] == n_H else fi['grad'][idx_H]
        for fi in fi_derivs_list if fi['grad'] is not None], axis=1)

if w2:
    HESS_H = np.stack([
    fi['hess'] if fi['hess'].shape[0] == n_H else fi['hess'][idx_H]
    for fi in fi_derivs_list if fi['hess'] is not None], axis=1)

if w3:
    idx_with_third = [i for i, fi in enumerate(fi_derivs_list) if fi['third'] is not None]
    if idx_with_third:
        thirds = []
        for i in idx_with_third:
            t = fi_derivs_list[i]['third']
            thirds.append(t if t.shape[0] == n_H else t[idx_H])
        # THIRD_H : (n_H, n_with_third, d, d, d)
        THIRD_H = np.stack(thirds, axis=1)
        # Gram partielle sur les fonctions qui ont third
        G_partial = np.einsum('xiklm,xjklm->ij', THIRD_H, THIRD_H, optimize='optimal') * w3 / n_H
        # Insérer dans G_H aux bons indices
        for ii, i in enumerate(idx_with_third):
            for jj, j in enumerate(idx_with_third):
                G_H[i, j] += G_partial[ii, jj]

print(f"  G_H calculée ({n_H} pts)")

# ── SÉLECTION DES EXPERTS PAR LOSS ────────────────────────────────────────────
# Loss du problème de Phase 2 restreint au sous-espace de dim 1 engendré par f_i
# (même train, même G_H, même λ_H) :
#   loss_i = min_c (1/n)||y - c f_i||² + λ_H c² G_H[i,i]
# Solution fermée : c_i* = m_i/(q_i + λ_H G_H[i,i]) avec m_i = <f_i,y>/n,
# q_i = ||f_i||²_n, et  loss_i = mean(y²) - m_i²/(q_i + λ_H G_H[i,i]).
# Chaque expert est ainsi optimalement rescalé avant comparaison.
n_loc  = len(y_train)
m_vec  = F_train.T @ y_train / n_loc
q_vec  = np.sum(F_train**2, axis=0) / n_loc
denom  = q_vec + params["lambda_reg"] * np.diag(G_H)
c_opt  = m_vec / denom
expert_losses = np.mean(y_train**2) - m_vec**2 / denom
best_loss   = expert_losses.min()
sel         = expert_losses <= params["k_loss"] * best_loss
sel_idx     = np.where(sel)[0]

print(f"\n  Sélection experts (k_loss={params['k_loss']}, "
      f"seuil={params['k_loss']*best_loss:.3e}) :")
for i in range(n_lev):
    tag = "GARDÉ " if sel[i] else "écarté"
    print(f"    {expert_names[i]:7s} : loss={expert_losses[i]:.3e}  c*={c_opt[i]:+.3f}  [{tag}]")
print(f"  -> {sel.sum()}/{n_lev} experts retenus")

F_train_sel = F_train[:, sel]
F_test_sel  = F_test[:,  sel]
G_H_sel     = G_H[np.ix_(sel_idx, sel_idx)]

# Résolution sur les experts sélectionnés :
# (F^T F / n + λ_H G_H) α = F^T y / n
n_tr = F_train_sel.shape[0]
M_H  = (F_train_sel.T@F_train_sel)/n_tr + params["lambda_reg"]*G_H_sel
rhs_H = (F_train_sel.T@y_train)/n_tr

eigvals_H,eigvecs_H = np.linalg.eigh(M_H)
thresh_H = max(params["lambda_reg"], 1e-10)
mask_H   = eigvals_H > thresh_H
V_H=eigvecs_H[:,mask_H]; S_H=eigvals_H[mask_H]
alpha_sel = V_H@((V_H.T@rhs_H)/S_H)

# alpha complet (0 sur les experts écartés) pour la suite du script
alpha = np.zeros(n_lev); alpha[sel_idx] = alpha_sel

pred_H_train = F_train_sel @ alpha_sel
pred_H_test  = F_test_sel  @ alpha_sel
mse_H = mean_squared_error(y_test, pred_H_test)
mse_H_train = mean_squared_error(y_train, pred_H_train)

print(f"  rang H = {mask_H.sum()}/{sel.sum()} (sur {n_lev} experts)")
print(f"  MSE train = {mse_H_train:.6f}  MSE test = {mse_H:.6f}")
print(f"  alpha = {alpha.round(3)}")


# ── MSE de H sans les experts RBF et Ridge ───────────────────────────────────
F_train_solo = F_train[:, :n_levels_total]
F_test_solo  = F_test[:,  :n_levels_total]
G_H_solo     = G_H[:n_levels_total, :n_levels_total]

M_solo  = (F_train_solo.T @ F_train_solo) / n_tr + params["lambda_reg"] * G_H_solo
rhs_solo = (F_train_solo.T @ y_train) / n_tr
ev_s, ec_s = np.linalg.eigh(M_solo)
mask_s = ev_s > max(params["lambda_reg"], 1e-10)
alpha_solo = ec_s[:, mask_s] @ ((ec_s[:, mask_s].T @ rhs_solo) / ev_s[mask_s])
mse_H_solo = mean_squared_error(y_test, F_test_solo @ alpha_solo)
print(f"Sobolev sous-espace H (sans experts) : {mse_H_solo:.6f}  (rang={mask_s.sum()})")
# ══════════════════════════════════════════════════════════════════════════════
# COMPARAISONS
# ══════════════════════════════════════════════════════════════════════════════
from sklearn.kernel_ridge import KernelRidge
from sklearn.preprocessing import PolynomialFeatures
from sklearn.model_selection import GridSearchCV

print("\nCalibration RBF (baseline de comparaison uniquement, hors H)...")
gamma_grid = np.logspace(-3,2,20)
param_grid = {'alpha':[1e-8,1e-5,1e-3],'gamma':gamma_grid,'kernel':['rbf']}
kr_cv = GridSearchCV(KernelRidge(), param_grid=param_grid, cv=5,
                     scoring='neg_mean_squared_error', n_jobs=-1)
kr_cv.fit(X_train, y_train)
best_gamma = kr_cv.best_params_['gamma']
best_alpha = kr_cv.best_params_['alpha']
best_err_rbf = mean_squared_error(y_test, kr_cv.predict(X_test))
results = kr_cv.cv_results_; order = np.argsort(results['rank_test_score'])
print("\nTop 3 RBF :")
for i in range(3):
    idx = order[i]; g = results['param_gamma'][idx]; a = results['param_alpha'][idx]
    kr = KernelRidge(kernel='rbf', gamma=float(g), alpha=float(a))
    kr.fit(X_train, y_train)
    print(f"  #{i+1} γ={g:.4f} α={a:.0e} TEST={mean_squared_error(y_test, kr.predict(X_test)):.6f}")

poly=PolynomialFeatures(degree=min(params["deg_P"],8),include_bias=False)
ridge_poly=Ridge(alpha=1e-8); ridge_poly.fit(poly.fit_transform(X_train),y_train)
err_poly=mean_squared_error(y_test,ridge_poly.predict(poly.transform(X_test)))

# ══════════════════════════════════════════════════════════════════════════════
# RÉSUMÉ
# ══════════════════════════════════════════════════════════════════════════════
print("\n"+"="*70); print("RÉSUMÉ"); print("="*70)
print(f"Polynomial Ridge  : {err_poly:.6f}")
print(f"RBF (γ={best_gamma:.4f})  : {best_err_rbf:.6f}")
print(f"\nMSE individuelles des experts :")
for i in range(n_lev):
    mse_i = mean_squared_error(y_test, F_test[:,i])
    tag = "" if sel[i] else "  (écarté)"
    print(f"  {expert_names[i]:7s} : {mse_i:.6f}  loss={expert_losses[i]:.3e}{tag}")
print(f"\nSobolev sous-espace H : {mse_H:.6f}  (rang={mask_H.sum()})")
print(f"\nweights={params['weights']}  λ={params['lambda_reg']} (unique)")
print(f"n_levels={params['n_levels']} (+{params['n_levels_iso']} iso)  "
      f"n_centres={params['n_centres']}  n_dict={params['n_dict']}  "
      f"k_loss={params['k_loss']} ({sel.sum()}/{n_lev} experts retenus)")
print(f"n_G={params['n_G']}  n_G_H={params['n_G_H']}")
print(f"σ∈[{params['sigma_min']},{params['sigma_max']}]")
print(f"temps phase 1 : {sum(times_p1):.1f}s total  ({np.mean(times_p1):.1f}s/niveau)")

from sklearn.metrics import roc_auc_score, accuracy_score

seuil = np.median(y_test)
y_test_class  = (y_test  > seuil).astype(int)
y_train_class = (y_train > seuil).astype(int)

pred_H     = F_test  @ alpha
pred_rbf = kr_cv.predict(X_test)
pred_ridge = ridge_poly.predict(poly.transform(X_test))

print("\n" + "="*70)
print("CLASSIFICATION (seuil = médiane de y_test)")
print("="*70)
for nom, pred in [("Sobolev H", pred_H), ("RBF", pred_rbf), ("Ridge poly", pred_ridge)]:
    pred_class = (pred > seuil).astype(int)
    acc = accuracy_score(y_test_class, pred_class)
    try:
        auc = roc_auc_score(y_test_class, pred)
    except Exception:
        auc = float('nan')
    print(f"  {nom:12s} : accuracy={acc:.4f}  AUC={auc:.4f}")

Génération du cloud sur {Q=0}...
  collectés:101/20 (50.5%)
  collectés:10083/5000 (50.4%)
  collectés:8080/3980 (50.8%)
  X_train:(20, 4)  X_unlabeled:(3980, 4)

Phase 1 : calcul de 10 fonctions indépendantes (5 cyl + 5 iso)...
  cyl_01  | rang=134/800 | loss=0.001100 (data=0.000072 reg=0.001028) | MSE=0.036973 | σ∈[0.10,3.00] | 25.4s
  cyl_02  | rang=145/800 | loss=0.001087 (data=0.000064 reg=0.001023) | MSE=0.038766 | σ∈[0.10,3.00] | 23.9s
  cyl_03  | rang=143/800 | loss=0.001070 (data=0.000063 reg=0.001007) | MSE=0.037977 | σ∈[0.10,2.99] | 25.1s
  cyl_04  | rang=140/800 | loss=0.001093 (data=0.000068 reg=0.001025) | MSE=0.039264 | σ∈[0.10,2.99] | 24.4s
  cyl_05  | rang=149/800 | loss=0.001075 (data=0.000061 reg=0.001014) | MSE=0.036393 | σ∈[0.11,3.00] | 39.3s
  iso_01  | rang= 10/800 | loss=0.002934 (data=0.001929 reg=0.001005) | MSE=0.055102 | σ∈[1.26,3.00] | 26.3s
  iso_02  | rang= 11/800 | loss=0.001769 (data=0.000653 reg=0.001116) | MSE=0.037336 | σ∈[1.19,2.99] | 25.3s
  iso_03